# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [54]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


In [55]:
DATA_DIR / "data" / "customer.csv"



PosixPath('/Users/atul/workspace/college_workspace/ban_6003/ban6003-final_project-option_2/data/data/customer.csv')

## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [56]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

In [57]:
customers.shape

(25000, 5)

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [58]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



items: 28,343 rows x 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.9,12.69
2,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.9,11.85


... (8 outputs left out)


geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [59]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [60]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

- **Business or research question:** 
  - Among Olist orders, which orders are likely to receive a negative review (1-2 stars), and what order, product, seller, and delivery factors drive that outcome?
- **Why this question matters:** 
  - Negative reviews hurt seller reputation, repeat-purchase rates, and platform trust. If the platform can flag orders at elevated risk of a bad review early (e.g. before an order is shipped), customer support or sellers could intervene (proactive communication, faster shipping) before the review is written.
- **Dataset files used:** 
  - `orders.csv`, `order_items.csv`, `order_payments.csv`, `order_reviews.csv`, `products.csv`, `sellers.csv`, `customers.csv`, `product_category_name_translation.csv`, and optionally `geolocation_zip_prefix.csv` for regional features.
- **Row meaning of each important table:**
  - `orders`: one row per order, keyed by `order_id`.
  - `customers`: one row per order-specific customer record, keyed by `customer_id`; `customer_unique_id` can repeat across orders for the same person.
  - `order_items`: one row per numbered item line within an order, keyed by `order_id` + `order_item_id`; must be aggregated to `order_id` before joining.
  - `order_payments`: one row per payment transaction/method sequence within an order, keyed by `order_id` + `payment_sequential`; must be aggregated to `order_id`.
  - `order_reviews`: one row per review record; not guaranteed unique by `order_id` or `review_id`. Must be reduced to one score per `order_id` before use.
  - `products`: one row per product (13,344 rows), keyed by `product_id`.
  - `sellers`: one row per seller (2,234 rows), keyed by `seller_id`.
- **Initial row and column counts:** See the "Basic Data Profile" table above for the full shape of all 9 tables. Full dtype/missingness breakdown per table is in the "Milestone 1 starter checks" output below this section.
- **Early data quality issues:**
  - 203 of 25,000 orders (0.8%) have no review row at all — these are unobserved outcomes, not automatically "high score."
  - 58.6% of review rows have a missing `review_comment_message` and 88.3% have a missing `review_comment_title` — missingness here is informative, not random, so it should be flagged with an indicator rather than imputed.
  - 230 products are missing `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` together — one shared root cause (incomplete catalog listing), not independent gaps.
  - Delivery-date missingness in `orders` tracks `order_status` (shipped/canceled/processing/invoiced/approved/created orders lack `order_delivered_customer_date`), not random loss.
  - A small number of impossible/suspicious values: 47 orders have `order_delivered_carrier_date` before `order_purchase_timestamp`, 19 have `order_delivered_customer_date` before `order_approved_at`, 2 payments have `payment_value` of 0, and 1 product has a missing weight.
  - Duplicate full rows: none found in any table at the whole-row level; duplicate *keys* are expected and already confirmed in item/payment/review detail tables (see Key and Granularity Audit above).
- **Initial cleaning plan:**
  1. Collapse `order_reviews` to one row per `order_id` (e.g., most recent `review_creation_date`, or mean score when scores disagree) before defining the target.
  2. Drop or separately flag the 203 orders with no review row; do not impute a score for them.
  3. Add a `has_review_comment` boolean instead of imputing missing review text.
  4. Aggregate `order_items` and `order_payments` to `order_id` (sum price/freight, count items, sum payments, count installments) before joining to `orders`, per the data guide.
  5. Fix or flag the small set of orders with impossible date ordering (carrier/delivered dates before purchase/approval) rather than silently trusting them.
  6. Join `product_category_name_translation` onto `products` before aggregating to order level so categories are in English for interpretation.
  7. Validate row counts and key uniqueness after every merge (`validate=` argument in `pd.merge`) to catch accidental fan-out.
- **Possible target/outcome:** `negative_review` = 1 if the order's (deduplicated) review score is 1 or 2, else 0; defined only for the 24,797 orders with an observed review. Class balance is imbalanced (~14.5% of review rows are 1-2 stars), so evaluation in Milestone 3 will need precision/recall, not just accuracy.
- **Ethics, privacy, or governance concerns:** `customers.csv`/`sellers.csv` contain city/state and zip-code-prefix fields that are quasi-identifying when combined with order history; no names, emails, or full addresses are present, but results should still be reported at an aggregate/segment level rather than singling out identifiable individuals. Review text may contain personal complaints, product defects, or incidental personal information typed by customers; it should not be republished verbatim and should be excluded from the feature set for now. A model that flags "at-risk-of-negative-review" orders should be used to trigger supportive outreach, not to penalize customers or deprioritize their service, and should be checked for uneven error rates across seller regions/categories before any operational use.


In [61]:
# Milestone 1 code workspace

orders_total = orders["order_id"].nunique()
orders_with_review = reviews["order_id"].nunique()
orders_without_review = orders_total - orders_with_review

print("Total orders:", orders_total)
print("Orders with a review:", orders_with_review)
print("Orders with no review:", orders_without_review)


Total orders: 25000
Orders with a review: 24797
Orders with no review: 203


In [62]:
# Review score distribution. 1-2 stars will be the "negative review" target.
reviews["review_score"].value_counts().sort_index()


review_score
1     2831
2      793
3     2019
4     4780
5    14511
Name: count, dtype: int64

In [63]:
# Some orders have more than one review row, so order_reviews is not one row per order_id yet.
print(reviews["order_id"].duplicated().sum())


137


In [64]:
# Missing review text is common. This should be flagged, not dropped.
reviews[["review_comment_title", "review_comment_message"]].isna().mean()


review_comment_title      0.882650
review_comment_message    0.585947
dtype: float64

In [65]:
# Check missing product metadata.
product_meta_cols = [
    "product_category_name", "product_name_lenght",
    "product_description_lenght", "product_photos_qty",
]
products[product_meta_cols].isna().sum()


product_category_name         230
product_name_lenght           230
product_description_lenght    230
product_photos_qty            230
dtype: int64

In [66]:
# Missing delivery dates by order_status.
# delivered_late should only be computed for delivered orders.
orders.groupby("order_status")["order_delivered_customer_date"].apply(lambda s: s.isna().sum())


order_status
approved         1
canceled       151
created          1
delivered        0
invoiced        75
processing      71
shipped        309
unavailable    167
Name: order_delivered_customer_date, dtype: int64

In [67]:
# Suspicious: payments with a value of zero or less.
(payments["payment_value"] <= 0).sum()


np.int64(2)

In [68]:
# Suspicious: carrier delivery date earlier than the purchase date.
(orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]).sum()


np.int64(47)

In [69]:
# Suspicious: missing or zero/negative product weight.
(products["product_weight_g"].isna() | (products["product_weight_g"] <= 0)).sum()


np.int64(1)

In [70]:
# Quick summary of what was found above.
milestone1_checks = pd.Series({
    "orders_total": orders_total,
    "orders_without_review": orders_without_review,
    "negative_review_rows": (reviews["review_score"] <= 2).sum(),
    "orders_with_multiple_reviews": reviews["order_id"].duplicated().sum(),
    "products_missing_category": products["product_category_name"].isna().sum(),
    "payments_zero_value": (payments["payment_value"] <= 0).sum(),
    "orders_carrier_before_purchase": (
        orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
    ).sum(),
}, name="count")

milestone1_checks


orders_total                      25000
orders_without_review               203
negative_review_rows               3624
orders_with_multiple_reviews        137
products_missing_category           230
payments_zero_value                   2
orders_carrier_before_purchase       47
Name: count, dtype: int64

In [71]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
customer_id,0.00000
order_status,0.00000
order_purchase_timestamp,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


,missing_rate
customer_id,0.0
customer_unique_id,0.0
customer_zip_code_prefix,0.0
customer_city,0.0
customer_state,0.0


... (17 outputs left out)

,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,str
product_category_name_english,str


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- **Final or preliminary unit of analysis:** Preliminary. One row = one order (`orders.order_id`), consistent with the Milestone 1 target definition.
- **Primary key for one row:** `order_id`. Confirmed unique (25,000 rows in, 25,000 rows out, 0 duplicate `order_id`) after every merge below.
- **Tables aggregated before joining:**
  - `order_items` (joined to `products_enriched` + `sellers` first, then aggregated to `order_id`) -> `item_summary`.
  - `order_payments` -> `payment_summary`.
  - `order_reviews` -> `review_summary` (collapsed from 24,934 rows / 137 orders with >1 review down to one row per `order_id` using mean score and a comment-present flag, since picking one row arbitrarily would silently discard disagreement between reviews on the same order).
- **Tables joined directly:** `orders` (base table) <- `customers` enriched with `geolocation_zip_prefix` (`customer_geo`) <- `item_summary` <- `payment_summary` <- `review_summary`. `products` and `sellers` are joined at the item level (inside `item_detail`), not directly onto `orders`, because category/seller attributes only make sense before they are aggregated away.
- **Row-count validation after each major merge:** Every merge above uses `validate=` (`many_to_one` for item/customer enrichment, `one_to_one` for the final order-level joins) plus an explicit `assert` on row count and/or key uniqueness immediately after it runs, so a silent fan-out would raise instead of passing quietly. Final check: `abt` has 25,000 rows, 25,000 unique `order_id`, 0 duplicates.
- **New variables created:** `freight_ratio` (freight / item price, a proxy for shipping-cost burden), `is_multi_item`, `is_multi_seller` (order complexity flags), `approval_hours` (purchase-to-approval delay), `purchase_month`, `purchase_dayofweek`, `promised_delivery_days`, `delivery_days`, `delivered_late` (nullable, delivered orders only), and the Milestone 1 target `negative_review` (nullable Int64, 1 if the order's deduplicated review score is 1-2, defined only for the 24,797 orders with an observed review). `negative_review` rate is 14.5%, matching the Milestone 1 estimate.
- **Remaining data quality concerns:** `order_item_count`/`item_price_total`/etc. are missing for the 209 orders with no item row at all (mostly canceled/unavailable orders); `delivery_days`/`delivered_late` are missing for undelivered orders by construction, not by error; `negative_review` is missing for the 203 orders with no review row and must stay missing (not imputed) through Milestone 3. `delivery_days` and `delivered_late` describe a post-purchase outcome and must **not** be used as Milestone 3 predictors for a prediction made at/near purchase time — they are kept here for exploratory analysis and flagged as leakage risk in the data dictionary below.



In [72]:
# Milestone 2 code workspace
# Build the order-level ABT (one row per order_id). Target for Milestone 3: `negative_review`
# (review_score <= 2), matching the target defined in Milestone 1. Decision timing: flag an
# order at/near purchase time, before it ships, so review- and delivery-outcome fields are
# only known after the fact and are documented as leakage risk, not used as predictors here.

products_enriched = products.merge(
    translations,
    on="product_category_name",
    how="left",
    validate="many_to_one",
)
print(f"products_enriched rows: {len(products_enriched):,} (products rows: {len(products):,})")
assert len(products_enriched) == len(products), "category translation changed row count"

item_detail = (
    items
    .merge(
        products_enriched[["product_id", "product_category_name_english", "product_weight_g"]],
        on="product_id", how="left", validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]],
        on="seller_id", how="left", validate="many_to_one",
    )
)
print(f"item_detail rows: {len(item_detail):,} (items rows: {len(items):,})")
assert len(item_detail) == len(items), "product/seller enrichment changed row count"


products_enriched rows: 13,344 (products rows: 13,344)
item_detail rows: 28,343 (items rows: 28,343)


In [73]:
item_summary = (
    item_detail.groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name_english", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        product_weight_mean_g=("product_weight_g", "mean"),
    )
    .reset_index()
)
item_summary["freight_ratio"] = item_summary["freight_total"] / item_summary["item_price_total"]
item_summary["is_multi_item"] = (item_summary["order_item_count"] > 1).astype(int)
item_summary["is_multi_seller"] = (item_summary["seller_count"] > 1).astype(int)
print(f"item_summary rows (orders with >=1 item): {len(item_summary):,}")
assert item_summary["order_id"].is_unique, "item_summary has duplicate order_id"

payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
    )
    .reset_index()
)
print(f"payment_summary rows (orders with >=1 payment): {len(payment_summary):,}")
assert payment_summary["order_id"].is_unique, "payment_summary has duplicate order_id"

reviews_for_summary = reviews.assign(
    review_comment_present=reviews["review_comment_message"].notna().astype(int)
)
review_summary = (
    reviews_for_summary.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_comment_present=("review_comment_present", "max"),
    )
    .reset_index()
)
print(f"review_summary rows (orders with >=1 review): {len(review_summary):,}")
assert review_summary["order_id"].is_unique, "review_summary has duplicate order_id"

customer_geo = customers.merge(
    geo[[
        "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
        "geolocation_records"
    ]],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).drop(columns=["geolocation_zip_code_prefix"]).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records",
})
print(f"customer_geo rows: {len(customer_geo):,} (customers rows: {len(customers):,})")
assert len(customer_geo) == len(customers), "geolocation enrichment changed row count"


item_summary rows (orders with >=1 item): 24,791
payment_summary rows (orders with >=1 payment): 25,000
review_summary rows (orders with >=1 review): 24,797
customer_geo rows: 25,000 (customers rows: 25,000)


In [74]:
abt = (
    orders
    .merge(customer_geo, on="customer_id", how="left", validate="one_to_one")
    .merge(item_summary, on="order_id", how="left", validate="one_to_one")
    .merge(payment_summary, on="order_id", how="left", validate="one_to_one")
    .merge(review_summary, on="order_id", how="left", validate="one_to_one")
)
assert len(abt) == len(orders), "merge changed row count - check join keys"
assert abt["order_id"].is_unique, "duplicate order_id after merges"
print(f"abt rows: {len(abt):,} (orders rows: {len(orders):,}), unique order_id: {abt['order_id'].nunique():,}")

abt["purchase_month"] = abt["order_purchase_timestamp"].dt.month
abt["purchase_dayofweek"] = abt["order_purchase_timestamp"].dt.dayofweek
abt["approval_hours"] = (
    abt["order_approved_at"] - abt["order_purchase_timestamp"]
).dt.total_seconds() / 3600
abt["promised_delivery_days"] = (
    abt["order_estimated_delivery_date"] - abt["order_purchase_timestamp"]
).dt.days

abt["delivery_days"] = (
    abt["order_delivered_customer_date"] - abt["order_purchase_timestamp"]
).dt.days

delivery_outcome_observed = (
    abt["order_status"].eq("delivered")
    & abt["order_delivered_customer_date"].notna()
    & abt["order_estimated_delivery_date"].notna()
)
abt["delivered_late"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[delivery_outcome_observed, "delivered_late"] = (
    abt.loc[delivery_outcome_observed, "order_delivered_customer_date"]
    > abt.loc[delivery_outcome_observed, "order_estimated_delivery_date"]
).astype(int)

abt["negative_review"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
review_outcome_observed = abt["review_score_mean"].notna()
abt.loc[review_outcome_observed, "negative_review"] = (
    abt.loc[review_outcome_observed, "review_score_mean"] <= 2
).astype(int)


abt rows: 25,000 (orders rows: 25,000), unique order_id: 25,000


In [75]:
abt_validation = pd.Series({
    "orders_before_merges": len(orders),
    "abt_rows_after_merges": len(abt),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": abt["order_id"].duplicated().sum(),
    "orders_with_item_summary": abt["order_item_count"].notna().sum(),
    "orders_with_payment_summary": abt["payment_total"].notna().sum(),
    "orders_with_review_summary": abt["review_score_mean"].notna().sum(),
    "orders_with_observed_delivery_target": abt["delivered_late"].notna().sum(),
    "orders_with_observed_review_target": abt["negative_review"].notna().sum(),
    "negative_review_rate": round(abt["negative_review"].mean(), 4),
})

display(abt_validation.to_frame("value"))
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


,value
orders_before_merges,25000.0000
abt_rows_after_merges,25000.0000
unique_order_ids,25000.0000
duplicate_order_ids,0.0000
orders_with_item_summary,24791.0000
orders_with_payment_summary,25000.0000
orders_with_review_summary,24797.0000
orders_with_observed_delivery_target,24222.0000
orders_with_observed_review_target,24797.0000
negative_review_rate,0.1449


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng,customer_geo_source_records,order_item_count,product_count,seller_count,category_count,seller_state_count,item_price_total,freight_total,product_weight_mean_g,freight_ratio,is_multi_item,is_multi_seller,payment_record_count,payment_total,payment_installments_max,payment_type_count,review_record_count,review_score_mean,review_comment_present,purchase_month,purchase_dayofweek,approval_hours,promised_delivery_days,delivery_days,delivered_late,negative_review
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,-23.581451,-46.635029,45.0,1.0,1.0,1.0,1.0,1.0,29.90,15.56,300.0,0.520401,0.0,0.0,1,45.46,1,1,1.0,4.0,1.0,10,0,78.101111,23,23.0,0,0
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,87776adb449c551e74c13fc34f036105,12030,taubate,SP,-23.032142,-45.570461,218.0,1.0,1.0,1.0,1.0,1.0,29.99,10.96,9000.0,0.365455,0.0,0.0,1,40.95,4,1,1.0,5.0,0.0,10,0,11.797778,50,10.0,0,0
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24,823c47d4abda1f8ce7568145f76c2b85,13175,sumare,SP,-22.817992,-47.241101,173.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,50.74,3,1,1.0,2.0,0.0,10,1,0.348611,50,NaN,<NA>,1
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28,0829f7df6577d5a4b65439bea701405f,31210,belo horizonte,MG,-19.906712,-43.948897,156.0,1.0,1.0,1.0,1.0,1.0,249.90,17.59,610.0,0.070388,0.0,0.0,1,267.49,10,1,1.0,3.0,0.0,10,1,0.521667,54,12.0,0,0
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28,df2988ba3ed226b10521a0e4da849b61,23092,rio de janeiro,RJ,-22.870856,-43.543849,217.0,1.0,1.0,1.0,1.0,1.0,67.90,18.98,2100.0,0.279529,0.0,0.0,7,86.88,1,1,1.0,5.0,1.0,10,1,0.413611,54,51.0,0,0


,missing_rate
delivered_late,0.03112
delivery_days,0.03100
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
product_weight_mean_g,0.00844
item_price_total,0.00836
freight_ratio,0.00836
product_count,0.00836
order_item_count,0.00836
seller_state_count,0.00836


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the order-level ABT row count, key uniqueness, and target eligibility without creating another permanent database file.


In [76]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("olist_order_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            SUM(CASE WHEN delivered_late IS NOT NULL THEN 1 ELSE 0 END) AS delivery_target_rows,
            SUM(CASE WHEN negative_review IS NOT NULL THEN 1 ELSE 0 END) AS review_target_rows,
            ROUND(AVG(CASE WHEN negative_review IS NOT NULL THEN negative_review END), 4) AS negative_review_rate
        FROM olist_order_abt
        ''',
        connection,
    )

display(sql_abt_check)


,abt_rows,unique_order_ids,delivery_target_rows,review_target_rows,negative_review_rate
0,25000,25000,24222,24797,0.1449


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [77]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

doc_fields = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]

# One entry per ABT column. The four values follow the order of doc_fields:
# description, source, transformation_or_derivation, notes_or_concerns.
column_entries = {
    "order_id": [
        "Unique identifier for one order; the ABT primary key.", "orders",
        "None, carried as-is.",
        "Unique by construction; validated after every merge above.",
    ],
    "customer_id": [
        "Identifier for the customer record tied to this one order.", "orders",
        "None, carried as-is.",
        "Not the same as customer_unique_id; one person can place multiple orders under different customer_id values.",
    ],
    "order_status": [
        "Order lifecycle status (e.g. delivered, shipped, canceled).", "orders",
        "None, carried as-is.",
        "Drives which rows have an observed delivered_late target; only 'delivered' orders with both dates set get a target value.",
    ],
    "order_purchase_timestamp": [
        "Timestamp the order was placed.", "orders",
        "None, carried as-is.",
        "Anchor timestamp for all derived time features below; defines the prediction-time cutoff for Milestone 3.",
    ],
    "order_approved_at": [
        "Timestamp the order payment was approved.", "orders",
        "None, carried as-is.",
        "0.16% missing; used only to derive approval_hours.",
    ],
    "order_delivered_carrier_date": [
        "Timestamp the order was handed to the logistics carrier.", "orders",
        "None, carried as-is.",
        "1.8% missing for undelivered/canceled orders; 47 rows have this before order_purchase_timestamp (flagged in Milestone 1, not yet corrected).",
    ],
    "order_delivered_customer_date": [
        "Timestamp the order reached the customer.", "orders",
        "None, carried as-is.",
        "3.1% missing for undelivered orders by construction; post-purchase outcome, leakage risk for a purchase-time model.",
    ],
    "order_estimated_delivery_date": [
        "Delivery date estimate shown to the customer at purchase time.", "orders",
        "None, carried as-is.",
        "Available at purchase time, so it is safe to use directly or through promised_delivery_days.",
    ],
    "customer_unique_id": [
        "Person-level identifier that can repeat across multiple orders/customer_id values.", "customers",
        "None, carried as-is.",
        "236 repeated values in the full customer table; not unique at the order grain and not used as a join key here.",
    ],
    "customer_zip_code_prefix": [
        "First digits of the customer's zip code (coarse location).", "customers",
        "None, carried as-is.",
        "Join key used to attach approximate lat/lng from geolocation_zip_prefix.",
    ],
    "customer_city": [
        "Customer city, self-reported at the zip-prefix level.", "customers",
        "None, carried as-is.",
        "Free-text city names are not fully standardized (casing/spelling variants observed in Milestone 1 profiling).",
    ],
    "customer_state": [
        "Customer state (Brazilian state abbreviation).", "customers",
        "None, carried as-is.",
        "Candidate categorical feature for regional patterns in negative_review.",
    ],
    "customer_lat": [
        "Approximate latitude for the customer's zip-code prefix.", "geolocation_zip_prefix",
        "Joined on customer_zip_code_prefix = geolocation_zip_code_prefix.",
        "Zip-prefix level, not an exact address; missing when the prefix has no geolocation record.",
    ],
    "customer_lng": [
        "Approximate longitude for the customer's zip-code prefix.", "geolocation_zip_prefix",
        "Joined on customer_zip_code_prefix = geolocation_zip_code_prefix.",
        "Zip-prefix level, not an exact address; missing when the prefix has no geolocation record.",
    ],
    "customer_geo_source_records": [
        "Number of raw geolocation records averaged into this zip-prefix's lat/lng.", "geolocation_zip_prefix",
        "Joined on customer_zip_code_prefix = geolocation_zip_code_prefix.",
        "Low values mean the lat/lng estimate for that prefix is less stable.",
    ],
    "order_item_count": [
        "Number of item lines on the order.", "order_items",
        "COUNT of order_item_id grouped by order_id.",
        "Missing for the 209 orders with no item row (mostly canceled/unavailable orders).",
    ],
    "product_count": [
        "Number of distinct products on the order.", "order_items",
        "Distinct COUNT of product_id grouped by order_id.",
        "Equal to order_item_count unless the same product appears on more than one item line.",
    ],
    "seller_count": [
        "Number of distinct sellers fulfilling the order.", "order_items",
        "Distinct COUNT of seller_id grouped by order_id.",
        "Feeds is_multi_seller; a split-seller order is more complex to fulfill correctly.",
    ],
    "category_count": [
        "Number of distinct English product categories on the order.", "order_items + products + translations",
        "Distinct COUNT of product_category_name_english grouped by order_id.",
        "NaN category rows (230 products) are not counted, so this can undercount for those orders.",
    ],
    "seller_state_count": [
        "Number of distinct seller states shipping this order.", "order_items + sellers",
        "Distinct COUNT of seller_state grouped by order_id.",
        "Candidate signal for cross-region logistics complexity.",
    ],
    "item_price_total": [
        "Sum of item prices on the order (excludes freight).", "order_items",
        "SUM of price grouped by order_id.",
        "Missing for orders with no item row; same 209-order gap as order_item_count.",
    ],
    "freight_total": [
        "Sum of freight (shipping) charges on the order.", "order_items",
        "SUM of freight_value grouped by order_id.",
        "Used with item_price_total to build freight_ratio.",
    ],
    "product_weight_mean_g": [
        "Average product weight (grams) across the order's item lines.", "order_items + products",
        "MEAN of product_weight_g grouped by order_id.",
        "1 product in the catalog has a missing/invalid weight, which propagates here for the orders that include it.",
    ],
    "freight_ratio": [
        "Freight cost relative to item price (freight_total / item_price_total).", "derived",
        "freight_total divided by item_price_total.",
        "Undefined (NaN) when item_price_total is 0 or missing; a high ratio is a hypothesized complaint driver.",
    ],
    "is_multi_item": [
        "1 if the order has more than one item line, else 0.", "derived",
        "order_item_count > 1, cast to int.",
        "Simple complexity flag; correlated with order_item_count.",
    ],
    "is_multi_seller": [
        "1 if the order is fulfilled by more than one seller, else 0.", "derived",
        "seller_count > 1, cast to int.",
        "Simple complexity flag; correlated with seller_count.",
    ],
    "payment_record_count": [
        "Number of payment transaction rows on the order.", "order_payments",
        "COUNT of payment_sequential grouped by order_id.",
        "More than one row means the order was paid across multiple methods/installments entries.",
    ],
    "payment_total": [
        "Sum of all payment amounts recorded for the order.", "order_payments",
        "SUM of payment_value grouped by order_id.",
        "2 payment rows in the raw table are zero or negative (flagged in Milestone 1); not yet corrected here.",
    ],
    "payment_installments_max": [
        "Largest number of installments used across the order's payment rows.", "order_payments",
        "MAX of payment_installments grouped by order_id.",
        "High installment counts may indicate price-sensitive customers.",
    ],
    "payment_type_count": [
        "Number of distinct payment methods used on the order.", "order_payments",
        "Distinct COUNT of payment_type grouped by order_id.",
        "Rare to be >1; mostly a data-quality sanity check column.",
    ],
    "review_record_count": [
        "Number of raw review rows collapsed into this order's review summary.", "order_reviews",
        "COUNT of review_id grouped by order_id.",
        "137 orders have more than 1; review_score_mean smooths over any disagreement between them.",
    ],
    "review_score_mean": [
        "Mean of the order's review score(s), 1-5.", "order_reviews",
        "MEAN of review_score grouped by order_id.",
        "Missing for the 203 orders with no review row at all; source column for the negative_review target.",
    ],
    "review_comment_present": [
        "1 if at least one of the order's reviews has non-missing comment text, else 0.", "order_reviews",
        "MAX of (review_comment_message is not null) grouped by order_id.",
        "Missing text is common (58.6% of review rows) and is informative, not random; this flag is kept instead of imputing comment text.",
    ],
    "purchase_month": [
        "Calendar month of order_purchase_timestamp (1-12).", "derived",
        "order_purchase_timestamp.dt.month.",
        "Candidate seasonality feature.",
    ],
    "purchase_dayofweek": [
        "Day of week of order_purchase_timestamp (0=Monday).", "derived",
        "order_purchase_timestamp.dt.dayofweek.",
        "Candidate seasonality feature.",
    ],
    "approval_hours": [
        "Hours between order_purchase_timestamp and order_approved_at.", "derived",
        "(order_approved_at - order_purchase_timestamp) in hours.",
        "Missing when order_approved_at is missing (0.16% of orders); a very long delay may itself signal a troubled order.",
    ],
    "promised_delivery_days": [
        "Days between purchase and the delivery estimate shown to the customer.", "derived",
        "(order_estimated_delivery_date - order_purchase_timestamp).dt.days.",
        "Known at purchase time; safe to use as a Milestone 3 predictor.",
    ],
    "delivery_days": [
        "Days between purchase and actual delivery to the customer.", "derived",
        "(order_delivered_customer_date - order_purchase_timestamp).dt.days.",
        "LEAKAGE RISK: only known after delivery; exploratory/EDA use only, not a purchase-time predictor.",
    ],
    "delivered_late": [
        "1 if a delivered order arrived after its estimated delivery date, else 0; missing otherwise.", "derived",
        "order_delivered_customer_date > order_estimated_delivery_date, computed only where order_status == 'delivered' and both dates are present.",
        "LEAKAGE RISK for the negative_review target and not usable as a purchase-time predictor; kept as an alternate target/EDA variable. Missing for non-delivered orders by design.",
    ],
    "negative_review": [
        "Milestone 3 target: 1 if the order's review_score_mean is 1-2, else 0; missing if unobserved.", "derived",
        "review_score_mean <= 2, computed only where review_score_mean is not null.",
        "Missing for the 203 orders with no review row; must stay missing through Milestone 3, not be imputed as 0. Base rate 14.5%, so evaluation needs precision/recall, not just accuracy.",
    ],
}

for abt_column, values in column_entries.items():
    data_dictionary.loc[data_dictionary["column_name"] == abt_column, doc_fields] = values

undocumented = data_dictionary.loc[
    data_dictionary[doc_fields].eq("").any(axis=1), "column_name"
].tolist()
print("ABT columns without a complete dictionary row:", undocumented)

data_dictionary


ABT columns without a complete dictionary row: []


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,order_id,str,Unique identifier for one order; the ABT prima...,orders,"None, carried as-is.",Unique by construction; validated after every ...
1,customer_id,str,Identifier for the customer record tied to thi...,orders,"None, carried as-is.",Not the same as customer_unique_id; one person...
2,order_status,str,"Order lifecycle status (e.g. delivered, shippe...",orders,"None, carried as-is.",Drives which rows have an observed delivered_l...
3,order_purchase_timestamp,datetime64[us],Timestamp the order was placed.,orders,"None, carried as-is.",Anchor timestamp for all derived time features...
4,order_approved_at,datetime64[us],Timestamp the order payment was approved.,orders,"None, carried as-is.",0.16% missing; used only to derive approval_ho...
5,order_delivered_carrier_date,datetime64[us],Timestamp the order was handed to the logistic...,orders,"None, carried as-is.",1.8% missing for undelivered/canceled orders; ...
6,order_delivered_customer_date,datetime64[us],Timestamp the order reached the customer.,orders,"None, carried as-is.",3.1% missing for undelivered orders by constru...
7,order_estimated_delivery_date,datetime64[us],Delivery date estimate shown to the customer a...,orders,"None, carried as-is.","Available at purchase time, so it is safe to u..."
8,customer_unique_id,str,Person-level identifier that can repeat across...,customers,"None, carried as-is.",236 repeated values in the full customer table...
9,customer_zip_code_prefix,int64,First digits of the customer's zip code (coars...,customers,"None, carried as-is.",Join key used to attach approximate lat/lng fr...


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [78]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [79]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,25000,25000,3b697a20d9e427646d92567910af6d57,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,25000,25000,355077684019f7f60a031656bd7262b8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,25000,8,delivered,24222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,25000,NaN,NaN,NaN,2017-12-31 22:18:29.561480,2016-10-03 09:44:50,2017-09-13 11:52:50,2018-01-19 19:08:12.500000,2018-05-05 11:17:57,2018-10-01 15:30:09,NaN
order_approved_at,24961,NaN,NaN,NaN,2018-01-01 06:46:13.413765,2016-10-04 10:19:23,2017-09-13 19:55:11,2018-01-20 09:08:59,2018-05-05 11:50:40,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,24548,NaN,NaN,NaN,2018-01-05 06:50:25.146529,2016-10-08 10:34:01,2017-09-18 18:01:51.750000,2018-01-24 20:01:44.500000,2018-05-08 14:21:00,2018-09-04 15:25:00,NaN
order_delivered_customer_date,24225,NaN,NaN,NaN,2018-01-14 19:15:33.384024,2016-10-11 14:46:49,2017-09-26 19:04:55,2018-02-03 03:49:26,2018-05-16 12:26:50,2018-09-27 02:24:33,NaN
order_estimated_delivery_date,25000,NaN,NaN,NaN,2018-01-24 17:19:26.976000,2016-10-27 00:00:00,2017-10-04 00:00:00,2018-02-15 00:00:00,2018-05-28 00:00:00,2018-10-23 00:00:00,NaN
customer_unique_id,25000,24764,8d50f5eadf50201ccdcedfb9e2ac8455,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,25000.0,NaN,NaN,NaN,34966.35068,1004.0,11431.75,24326.0,58037.0,99970.0,29706.603087


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [80]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [81]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
